In [ ]:
# 1. 사전 환경 준비
!pip install shap


### Import & Data Load


In [ ]:
# Import & Data Load

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder

In [ ]:
import os
# 실행 경로와 상관없이 데이터를 로드할 수 있도록 상대 경로 자동 조정
data_dir = '.' if os.path.exists('train.csv') else '../..'
train = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test = pd.read_csv(os.path.join(data_dir, 'test.csv'))


In [ ]:
train.head(5)

### Check Data


In [ ]:
train.info()

In [ ]:
train.isnull().sum()

In [ ]:
# 결측값 있는 칼럼(column) 확인
missing_columns_train = train.columns[train.isnull().sum() > 0]
missing_columns_train

In [ ]:
train[missing_columns_train].info()

In [ ]:
# ==========================================
# 1. 통합 결측치 처리 및 3D Cascade 대치
# ==========================================
for df in [train, test]:
    df['medical_history'] = df['medical_history'].fillna('none')
    df['family_medical_history'] = df['family_medical_history'].fillna('none')
    df['edu_level'] = df['edu_level'].fillna('Unknown')
    df['is_working_missing'] = df['mean_working'].isnull().astype(int)

# Ordinal Encoding 매핑 사전 선언
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}
activity_map = {'light': 1, 'moderate': 2, 'intense': 3}
for df in [train, test]:
    df['edu_level_encoded'] = df['edu_level'].map(edu_map)
    df['activity_encoded'] = df['activity'].map(activity_map)

# 3D 세분화 결측 대치 그룹
train['age_group'] = (train['age'] // 10) * 10
test['age_group'] = (test['age'] // 10) * 10

group_cols_3d = ['age_group', 'edu_level_encoded', 'activity_encoded']
group_cols_2d = ['age_group', 'edu_level_encoded']

medians_3d = train.groupby(group_cols_3d)['mean_working'].median()
medians_2d = train.groupby(group_cols_2d)['mean_working'].median()
overall_median = train['mean_working'].median()

def impute_working(row, medians_3d, medians_2d, overall_median):
    if not pd.isnull(row['mean_working']):
        return row['mean_working']
    key_3d = (row['age_group'], row['edu_level_encoded'], row['activity_encoded'])
    if key_3d in medians_3d and not pd.isnull(medians_3d[key_3d]):
        return medians_3d[key_3d]
    key_2d = (row['age_group'], row['edu_level_encoded'])
    if key_2d in medians_2d and not pd.isnull(medians_2d[key_2d]):
        return medians_2d[key_2d]
    return overall_median

train['mean_working'] = train.apply(lambda r: impute_working(r, medians_3d, medians_2d, overall_median), axis=1)
test['mean_working'] = test.apply(lambda r: impute_working(r, medians_3d, medians_2d, overall_median), axis=1)

train.drop('age_group', axis=1, inplace=True)
test.drop('age_group', axis=1, inplace=True)

# ==========================================
# 2. 피처 정제 및 bmi 복원 (Ponderal Index, Hypertension, Risk Score 등 구간화/상관 변수 전면 삭제)
# ==========================================
for df in [train, test]:
    # 의학 표준 BMI 복원
    df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)
    
    # 비선형 생체 지표 도메인 결합
    df['pulse_pressure'] = df['systolic_blood_pressure'] - df['diastolic_blood_pressure']
    df['map'] = (df['systolic_blood_pressure'] + 2 * df['diastolic_blood_pressure']) / 3
    df['glucose_cholesterol_ratio'] = df['glucose'] / (df['cholesterol'] + 1)
    df['overwork_and_poor_sleep'] = ((df['mean_working'] >= 12) & (df['sleep_pattern'] == 'sleep difficulty')).astype(int)
    df['vascular_bone_risk'] = ((df['bone_density'] <= -1.0) & (df['pulse_pressure'] > 80)).astype(int)

# 복합 질환(medical_history) 동적 이진화 플래그 생성 (오직 train 기준으로만 질환 Vocabulary 추출 - Leakage 완전 차단)
diseases = set()
for col in ['medical_history', 'family_medical_history']:
    for val in train[col].dropna().unique():
        for d in val.split(','):
            diseases.add(d.strip())
diseases.discard('none')
diseases = sorted(list(diseases))

for col, prefix in [('medical_history', 'med'), ('family_medical_history', 'fam')]:
    for disease in diseases:
        feat_name = f'{prefix}_{disease.lower().replace(" ", "_")}'
        train[feat_name] = train[col].apply(lambda x: 1 if disease in [d.strip() for d in x.split(",")] else 0)
        test[feat_name] = test[col].apply(lambda x: 1 if disease in [d.strip() for d in x.split(",")] else 0)

# 사용이 끝난 원본 문자열 컬럼 일괄 삭제
drop_cols = ['edu_level', 'activity', 'medical_history', 'family_medical_history']
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)


In [ ]:
# 순서형 인코딩된 컬럼을 제외한 순수 범주형 데이터 변환 및 타입 지정
categorical_cols = ['gender', 'smoke_status', 'sleep_pattern']

for col in categorical_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')


In [ ]:
x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)


In [ ]:
import shap
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler

print("Calculating SHAP values for baseline model feature selection...")
x_train_temp = x_train.copy()
scaler_g = RobustScaler()
scaler_c = RobustScaler()
x_train_temp['metabolic_load_index'] = (scaler_g.fit_transform(x_train_temp[['glucose']]) + scaler_c.fit_transform(x_train_temp[['cholesterol']])).flatten()

# Baseline LightGBM 학습
baseline_model = lgb.LGBMRegressor(random_state=42, verbose=-1)
baseline_model.fit(x_train_temp, y_train)

# SHAP TreeExplainer 기여도 계산
explainer = shap.TreeExplainer(baseline_model)
shap_values = explainer.shap_values(x_train_temp)

if isinstance(shap_values, list):
    shap_avg = np.abs(shap_values[0]).mean(axis=0)
else:
    shap_avg = np.abs(shap_values).mean(axis=0)

shap_imp = pd.Series(shap_avg, index=x_train_temp.columns).sort_values(ascending=False)
print("\n--- SHAP Feature Importances (Absolute Mean) ---")
for feat, val in shap_imp.items():
    print(f"{feat}: {val:.6f}")

# summary_plot(bar) 출력 및 이미지 파일 저장
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, x_train_temp, plot_type='bar', show=False)
plt.tight_layout()
output_dir = 'result/v5' if os.path.exists('result/v5') else '.'
os.makedirs(output_dir, exist_ok=True)
plt.savefig(os.path.join(output_dir, 'shap_summary_plot.png'))
plt.close()
print(f"\nSHAP summary plot saved to {os.path.join(output_dir, 'shap_summary_plot.png')}")

# 최하위 N개 피처 자동 제거 (조절 변수 N = 3)
N = 3
features_to_drop = list(shap_imp.tail(N).index)
print(f"\nRemoving bottom {N} features from dataset: {features_to_drop}")

# x_train, x_test에서 제거
x_train = x_train.drop(columns=[f for f in features_to_drop if f in x_train.columns], errors='ignore')
x_test = x_test.drop(columns=[f for f in features_to_drop if f in x_test.columns], errors='ignore')
print(f"Pruned x_train shape: {x_train.shape}, pruned x_test shape: {x_test.shape}")


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import RobustScaler
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# dynamic cat_features list based on remaining columns after SHAP pruning
cat_cols = [c for c in ['gender', 'smoke_status', 'sleep_pattern'] if c in x_train.columns]
print(f"Categorical columns for CatBoost: {cat_cols}")

# 1. LightGBM 하이퍼파라미터 튜닝 (50 trials)
def tune_lgbm(x_train, y_train):
    def objective(trial):
        params = {
            'objective': 'regression_l1',
            'metric': 'mae',
            'random_state': 42,
            'verbose': -1,
            'n_jobs': 1,
            'n_estimators': trial.suggest_int('n_estimators', 200, 800),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 127),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        }
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        mae_scores = []
        for train_idx, val_idx in kf.split(x_train, y_train):
            X_tr, y_tr = x_train.iloc[train_idx].copy(), y_train.iloc[train_idx]
            X_val, y_val = x_train.iloc[val_idx].copy(), y_train.iloc[val_idx]
            
            scaler_g = RobustScaler()
            scaler_c = RobustScaler()
            X_tr['metabolic_load_index'] = (scaler_g.fit_transform(X_tr[['glucose']]) + scaler_c.fit_transform(X_tr[['cholesterol']])).flatten()
            X_val['metabolic_load_index'] = (scaler_g.transform(X_val[['glucose']]) + scaler_c.transform(X_val[['cholesterol']])).flatten()
            
            model = lgb.LGBMRegressor(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(30, verbose=False)])
            preds = model.predict(X_val)
            mae_scores.append(mean_absolute_error(y_val, preds))
        mean_mae = np.mean(mae_scores)
        print(f" => [LGBM] Trial {trial.number} Finished. Mean MAE: {mean_mae:.6f}")
        return mean_mae
    optuna.logging.set_verbosity(optuna.logging.INFO)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=50)
    return study.best_params

# 2. XGBoost 하이퍼파라미터 튜닝 (30 trials)
def tune_xgboost(x_train, y_train):
    def objective(trial):
        params = {
            'objective': 'reg:absoluteerror',
            'eval_metric': 'mae',
            'random_state': 42,
            'verbosity': 0,
            'n_jobs': 1,
            'enable_categorical': True,
            'tree_method': 'hist',
            'n_estimators': trial.suggest_int('n_estimators', 200, 800),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 9),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'alpha': trial.suggest_float('alpha', 1e-3, 10.0, log=True),
            'lambda': trial.suggest_float('lambda', 1e-3, 10.0, log=True),
        }
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        mae_scores = []
        for train_idx, val_idx in kf.split(x_train, y_train):
            X_tr, y_tr = x_train.iloc[train_idx].copy(), y_train.iloc[train_idx]
            X_val, y_val = x_train.iloc[val_idx].copy(), y_train.iloc[val_idx]
            
            scaler_g = RobustScaler()
            scaler_c = RobustScaler()
            X_tr['metabolic_load_index'] = (scaler_g.fit_transform(X_tr[['glucose']]) + scaler_c.fit_transform(X_tr[['cholesterol']])).flatten()
            X_val['metabolic_load_index'] = (scaler_g.transform(X_val[['glucose']]) + scaler_c.transform(X_val[['cholesterol']])).flatten()
            
            model = xgb.XGBRegressor(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            preds = model.predict(X_val)
            mae_scores.append(mean_absolute_error(y_val, preds))
        mean_mae = np.mean(mae_scores)
        print(f" => [XGBoost] Trial {trial.number} Finished. Mean MAE: {mean_mae:.6f}")
        return mean_mae
    import os
    optuna.logging.set_verbosity(optuna.logging.INFO)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=50)
    return study.best_params

# 3. CatBoost 하이퍼파라미터 튜닝 (50 trials, depth 4~8 범위 확장, CPU 기반)
def tune_catboost(x_train, y_train):
    def objective(trial):
        params = {
            'loss_function': 'MAE',
            'eval_metric': 'MAE',
            'random_seed': 42,
            'verbose': False,
            'task_type': 'CPU',
            'thread_count': 1,
            'n_estimators': trial.suggest_int('n_estimators', 200, 500),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'depth': trial.suggest_int('depth', 4, 8),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        }
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        mae_scores = []
        for train_idx, val_idx in kf.split(x_train, y_train):
            X_tr, y_tr = x_train.iloc[train_idx].copy(), y_train.iloc[train_idx]
            X_val, y_val = x_train.iloc[val_idx].copy(), y_train.iloc[val_idx]
            
            scaler_g = RobustScaler()
            scaler_c = RobustScaler()
            X_tr['metabolic_load_index'] = (scaler_g.fit_transform(X_tr[['glucose']]) + scaler_c.fit_transform(X_tr[['cholesterol']])).flatten()
            X_val['metabolic_load_index'] = (scaler_g.transform(X_val[['glucose']]) + scaler_c.transform(X_val[['cholesterol']])).flatten()
            
            model = CatBoostRegressor(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], cat_features=cat_cols, early_stopping_rounds=30, verbose=False)
            preds = model.predict(X_val)
            mae_scores.append(mean_absolute_error(y_val, preds))
        mean_mae = np.mean(mae_scores)
        print(f" => [CatBoost] Trial {trial.number} Finished. Mean MAE: {mean_mae:.6f}")
        return mean_mae
    import os
    optuna.logging.set_verbosity(optuna.logging.INFO)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=50)
    return study.best_params

print("Tuning LightGBM (50 trials)... ")
best_lgb = tune_lgbm(x_train, y_train)
print(f"Best LGBM Params: {best_lgb}")

print("Tuning XGBoost (50 trials)... ")
best_xgb = tune_xgboost(x_train, y_train)
print(f"Best XGBoost Params: {best_xgb}")

print("Tuning CatBoost (20 trials, depth 4-8)... ")
best_cat = tune_catboost(x_train, y_train)
print(f"Best CatBoost Params: {best_cat}")

# 4. Seed Averaging (Seeds: 42, 2026, 777) 최종 학습 수행 함수
def train_with_seeds(x_train, y_train, x_test, model_type, best_params, seeds=[42, 2026, 777]):
    oof_preds = np.zeros(len(x_train))
    test_preds = np.zeros(len(x_test))
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(x_train, y_train)):
        X_tr, y_tr = x_train.iloc[train_idx].copy(), y_train.iloc[train_idx]
        X_val, y_val = x_train.iloc[val_idx].copy(), y_train.iloc[val_idx]
        
        scaler_g = RobustScaler()
        scaler_c = RobustScaler()
        X_tr['metabolic_load_index'] = (scaler_g.fit_transform(X_tr[['glucose']]) + scaler_c.fit_transform(X_tr[['cholesterol']])).flatten()
        X_val['metabolic_load_index'] = (scaler_g.transform(X_val[['glucose']]) + scaler_c.transform(X_val[['cholesterol']])).flatten()
        
        x_te_fold = x_test.copy()
        x_te_fold['metabolic_load_index'] = (scaler_g.transform(x_te_fold[['glucose']]) + scaler_c.transform(x_te_fold[['cholesterol']])).flatten()
        
        fold_val_preds = np.zeros(len(X_val))
        fold_test_preds = np.zeros(len(x_te_fold))
        
        for seed in seeds:
            if model_type == 'lgb':
                params = best_params.copy()
                params.update({'objective': 'regression_l1', 'metric': 'mae', 'random_state': seed, 'verbose': -1, 'n_jobs': 1})
                model = lgb.LGBMRegressor(**params)
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(30, verbose=False)])
                
            elif model_type == 'xgb':
                params = best_params.copy()
                params.update({'objective': 'reg:absoluteerror', 'eval_metric': 'mae', 'random_state': seed, 'verbosity': 0, 'enable_categorical': True, 'tree_method': 'hist', 'n_jobs': 1})
                model = xgb.XGBRegressor(**params)
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
                
            elif model_type == 'cat':
                params = best_params.copy()
                params.update({'loss_function': 'MAE', 'eval_metric': 'MAE', 'random_seed': seed, 'verbose': False, 'task_type': 'CPU', 'thread_count': 1})
                model = CatBoostRegressor(**params)
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], cat_features=cat_cols, early_stopping_rounds=30, verbose=False)
                
            fold_val_preds += model.predict(X_val) / len(seeds)
            fold_test_preds += model.predict(x_te_fold) / len(seeds)
            
        oof_preds[val_idx] = fold_val_preds
        test_preds += fold_test_preds / 5
        
    mae = mean_absolute_error(y_train, oof_preds)
    print(f'[{model_type.upper()}] Seed Averaged OOF MAE: {mae:.6f}')
    return oof_preds, test_preds

print("Training models with seed averaging...")
oof_lgb, test_lgb = train_with_seeds(x_train, y_train, x_test, 'lgb', best_lgb)
oof_xgb, test_xgb = train_with_seeds(x_train, y_train, x_test, 'xgb', best_xgb)
oof_cat, test_cat = train_with_seeds(x_train, y_train, x_test, 'cat', best_cat)

# 5. SLSQP 가중치 최적화 블렌딩 (가중치 합 = 1.0, 0~1 바운드 설정)
oof_preds_list = [oof_lgb, oof_xgb, oof_cat]

def mae_objective(weights):
    w = weights / np.sum(weights)
    blended_pred = (w[0] * oof_preds_list[0]) + (w[1] * oof_preds_list[1]) + (w[2] * oof_preds_list[2])
    return mean_absolute_error(y_train, blended_pred)

constraints = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})
bounds = [(0, 1), (0, 1), (0, 1)]
initial_weights = [1/3, 1/3, 1/3]

res = minimize(mae_objective, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)
best_weights = res.x / np.sum(res.x)

print(f"\nOptimal Blending Weights (LGB, XGB, CAT): {best_weights}")
blended_oof = (best_weights[0] * oof_lgb) + (best_weights[1] * oof_xgb) + (best_weights[2] * oof_cat)

# Boundary Clipping 적용하여 안정성 보장
blended_oof_clipped = np.clip(blended_oof, 0.0, 1.0)
print(f"Final Blended Clipped OOF MAE: {mean_absolute_error(y_train, blended_oof_clipped):.6f}")

test_preds = (best_weights[0] * test_lgb) + (best_weights[1] * test_xgb) + (best_weights[2] * test_cat)
test_preds = np.clip(test_preds, 0.0, 1.0)


In [ ]:
submission = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))


In [ ]:
submission['stress_score'] = test_preds
submission.head()


In [ ]:
# 결과 저장 경로 자동 대응
output_dir = 'result/v5' if os.path.exists('result/v5') else '.'
submission.to_csv(os.path.join(output_dir, 'submit_05.csv'), index=False)
